# Colab — fine-tune YOLO11l từ Hugging Face

Bật GPU (Runtime → Change runtime type). Thêm HF_TOKEN vào Colab Secrets và cho phép notebook truy cập.
Upload dataset bằng scripts/upload_finetune_dataset.py trước. Mặc định repo ericng17/hcm-traffic-yolo-v20.
Mount Drive để lưu checkpoints qua các lần runtime ngắt; ảnh train tải về SSD /content.
Đây là fine-tune pretrained YOLO11l, không dùng best.pt cũ.


In [ ]:
%pip install "ultralytics>=8.3,<8.4" "huggingface_hub>=0.23" "Pillow>=10,<13" pyyaml


In [ ]:
import os, sys, json, zipfile
from pathlib import Path
from google.colab import userdata, drive
import torch, yaml
from huggingface_hub import HfApi, hf_hub_download
from ultralytics import YOLO

assert torch.cuda.is_available(), 'Bật GPU trước khi train'
print(torch.cuda.get_device_name(0))
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive')
REPO_ID = 'ericng17/hcm-traffic-yolo-v20'
DATA_ROOT = Path('/content/hcm-yolo-v20')
PROJECT = Path('/content/drive/MyDrive/tfp/yolo-training')
RUN_NAME = 'yolo11l-v20-run1'  # Tên mới cho thực nghiệm mới
RESUME = False               # True chỉ để tiếp tục run bị ngắt, cùng cấu hình/data
EPOCHS = 100
BATCH = 4                    # OOM: thử 2 ở run mới


In [ ]:
PROJECT.mkdir(parents=True, exist_ok=True)
provenance = PROJECT / f'{RUN_NAME}-source.json'
api = HfApi(token=os.environ['HF_TOKEN'])
if provenance.exists():
    source = json.loads(provenance.read_text())
    assert source['repo_id'] == REPO_ID
    revision = source['revision']
else:
    revision = api.repo_info(REPO_ID, repo_type='dataset').sha
    provenance.write_text(json.dumps({'repo_id': REPO_ID, 'revision': revision}, indent=2))
archive = hf_hub_download(REPO_ID, 'dataset.zip', repo_type='dataset', revision=revision)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for info in z.infolist():
        target = (DATA_ROOT / info.filename).resolve()
        assert target.is_relative_to(DATA_ROOT.resolve()), 'Unsafe ZIP path'
    z.extractall(DATA_ROOT)
cfg = yaml.safe_load((DATA_ROOT / 'data.yaml').read_text())
cfg.update(path=str(DATA_ROOT), train='train/images', val='valid/images', test='test/images')
(DATA_ROOT / 'data.yaml').write_text(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))
print(yaml.safe_dump(cfg, sort_keys=False))


## Kiểm tra lại định dạng và ảnh trước train

In [ ]:
"""Audit YOLO detection labels, image integrity and exact split overlap."""
import argparse
import hashlib
import json
import math
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from PIL import Image


def audit(root):
    root = Path(root).resolve()
    cfg = yaml.safe_load((root / 'data.yaml').read_text())
    names = cfg['names']
    if isinstance(names, dict):
        names = [names[i] for i in range(len(names))]
    errors, warnings, splits = [], [], {}
    if cfg.get('nc', len(names)) != len(names):
        errors.append('nc differs from names length')
    hashes = defaultdict(list)
    for split, folder in [('train', 'train'), ('val', 'valid'), ('test', 'test')]:
        image_dir, label_dir = root / folder / 'images', root / folder / 'labels'
        if not image_dir.is_dir() or not label_dir.is_dir():
            errors.append(f'{split}: missing images/labels directory')
            continue
        configured = (root / cfg[split]).resolve()
        if configured != image_dir:
            errors.append(f'{split}: YAML resolves to {configured}, expected {image_dir}')
        images = sorted(p for p in image_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        counts, sizes = Counter(), Counter()
        empty, missing, boxes, duplicate_rows = 0, 0, 0, 0
        stems = Counter(p.stem for p in images)
        for stem, count in stems.items():
            if count > 1:
                errors.append(f'{split}: multiple images share label stem {stem}')
        for image in images:
            relative = image.relative_to(root).as_posix()
            try:
                with Image.open(image) as im:
                    im.load()
                    width, height = im.size
                    sizes[f'{width}x{height}'] += 1
            except Exception as exc:
                errors.append(f'{relative}: unreadable image: {exc}')
                continue
            hashes[hashlib.sha256(image.read_bytes()).hexdigest()].append(relative)
            label = label_dir / (image.stem + '.txt')
            if not label.exists():
                missing += 1
                errors.append(f'{relative}: missing label (review before treating as background)')
                continue
            lines = [s for s in label.read_text().splitlines() if s.strip()]
            empty += not lines
            seen = set()
            for line_no, line in enumerate(lines, 1):
                location = f'{label.relative_to(root)}:{line_no}'
                try:
                    fields = line.split()
                    if len(fields) != 5:
                        raise ValueError('expected class xc yc w h (5 fields)')
                    cls, x, y, w, h = map(float, fields)
                    if not all(math.isfinite(v) for v in [cls, x, y, w, h]):
                        raise ValueError('non-finite value')
                    if not cls.is_integer() or not 0 <= cls < len(names):
                        raise ValueError('invalid class ID')
                    if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                        raise ValueError('invalid normalized xywh')
                    if x-w/2 < -1e-5 or y-h/2 < -1e-5 or x+w/2 > 1+1e-5 or y+h/2 > 1+1e-5:
                        raise ValueError('box extends outside image')
                    row = (cls, x, y, w, h)
                    duplicate_rows += row in seen
                    seen.add(row)
                    counts[names[int(cls)]] += 1
                    boxes += 1
                except ValueError as exc:
                    errors.append(f'{location}: {exc}')
        orphan = [str(p.relative_to(root)) for p in label_dir.glob('*.txt') if p.stem not in stems]
        errors.extend(f'orphan label: {p}' for p in orphan)
        if duplicate_rows:
            warnings.append(f'{split}: {duplicate_rows} duplicate annotation rows')
        if empty:
            warnings.append(f'{split}: {empty} empty labels; manually verify background images')
        splits[split] = dict(images=len(images), boxes=boxes, class_counts=dict(counts),
                             empty_labels=empty, missing_labels=missing, image_sizes=dict(sizes),
                             duplicate_label_rows=duplicate_rows)
    duplicates = [paths for paths in hashes.values() if len(paths) > 1]
    cross_split = [paths for paths in duplicates if len({p.split('/')[0] for p in paths}) > 1]
    if cross_split:
        warnings.append(f'{len(cross_split)} exact-image hash groups cross dataset splits')
    return dict(names=names, splits=splits, errors=errors, warnings=warnings,
                exact_duplicate_groups=duplicates, cross_split_duplicates=cross_split,
                scope='Format/integrity only; no semantic label or near-duplicate verification')



report = audit(DATA_ROOT)
print(json.dumps(report['splits'], indent=2))
print('Warnings:', report['warnings'])
assert not report['errors'], report['errors'][:20]


In [ ]:
# Xem annotation gốc (không phải dự đoán) trên ảnh từng split.
from PIL import Image, ImageDraw
from IPython.display import display
for split in ['train', 'valid', 'test']:
    candidates = sorted((DATA_ROOT / split / 'images').glob('*'))
    for p in candidates[:2]:
        image = Image.open(p).convert('RGB')
        draw = ImageDraw.Draw(image)
        w, h = image.size
        label = DATA_ROOT / split / 'labels' / (p.stem + '.txt')
        for line in label.read_text().splitlines():
            if not line.strip(): continue
            cls, x, y, bw, bh = map(float, line.split())
            box = ((x-bw/2)*w, (y-bh/2)*h, (x+bw/2)*w, (y+bh/2)*h)
            draw.rectangle(box, outline='lime', width=2)
            draw.text(box[:2], cfg['names'][int(cls)], fill='yellow')
        print(split, p.name)
        display(image)


In [ ]:
import ultralytics
(PROJECT / f'{RUN_NAME}-environment.json').write_text(json.dumps({
    'ultralytics': ultralytics.__version__, 'torch': torch.__version__,
    'gpu': torch.cuda.get_device_name(0), 'dataset_revision': revision,
}, indent=2))
last = PROJECT / RUN_NAME / 'weights/last.pt'
if RESUME:
    assert last.exists(), 'Không có last.pt để resume'
    model = YOLO(str(last))
    model.train(resume=True)
else:
    assert not (PROJECT / RUN_NAME).exists(), 'Đổi RUN_NAME hoặc chọn RESUME=True'
    model = YOLO('yolo11l.pt')
    model.train(data=str(DATA_ROOT / 'data.yaml'), imgsz=1024,
                epochs=EPOCHS, patience=20, batch=BATCH, device=0,
                workers=2, cache=False, seed=42, deterministic=True,
                mixup=0.0, close_mosaic=10, save=True, save_period=10,
                project=str(PROJECT), name=RUN_NAME)


## Validation và test cuối

Dùng validation để lựa chọn cấu hình. Chỉ chạy test khi đã chốt phương pháp; không dùng
kết quả test để chọn threshold/model. Test này vẫn là split dataset gốc, không thay thế
đánh giá trên 28 camera của bạn. Checkpoint last.pt được lưu cuối epoch; runtime mất
trước epoch đầu có thể chưa có checkpoint để resume.


In [ ]:
best = YOLO(str(PROJECT / RUN_NAME / 'weights/best.pt'))
metrics = best.val(data=str(DATA_ROOT / 'data.yaml'), split='val', imgsz=1024,
                   batch=BATCH, device=0, max_det=1000,
                   project=str(PROJECT), name=RUN_NAME+'-val')
print(metrics.results_dict)
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    test_metrics = best.val(data=str(DATA_ROOT / 'data.yaml'), split='test',
                            imgsz=1024, batch=BATCH, device=0, max_det=1000,
                            project=str(PROJECT), name=RUN_NAME+'-test')
    print(test_metrics.results_dict)
print('Weights đã lưu trên Drive:', PROJECT / RUN_NAME / 'weights')


Tài liệu: https://docs.ultralytics.com/modes/train/ và https://docs.ultralytics.com/datasets/detect/
